In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re, torch, seaborn
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

in_dir = f'{data_dir}/save_hiddens/zero_shot_{zero_shot_target}'
out_dir = f'{data_dir}/visualize_hiddens/zero_shot_{zero_shot_target}'

In [ ]:
def get_file_path_pca(out_prefix, n_comp, file_type, layer_idx):
    out_file_path = f'{out_prefix}/pca_{n_comp}d/{file_type}/pca_{n_comp}d_layer_{layer_idx}.{file_type}'
    file_utils.make_parent(out_file_path)

    return out_file_path

In [ ]:
def visualize_pca_2d(layer_idx, df, explained_var, out_prefix, do_show=True):
    # 1. 상세 확인용 인터랙티브 HTML (Plotly)
    fig_html = px.scatter(
        df, x='PC1', y='PC2', color='Group',
        color_discrete_map={'Fact': 'blue', 'Counter': 'red'},
        opacity=0.7,
        title=f'PCA 2D - Layer {layer_idx} (Top 2 Var: {sum(explained_var[:2]):.2f}%)',
        labels={
            'PC1': f'PC1 ({explained_var[0]:.2f}%)',
            'PC2': f'PC2 ({explained_var[1]:.2f}%)'
        },
        width=800,
        height=600
    )
    fig_html.write_html(get_file_path_pca(out_prefix, 2, 'html', layer_idx))

    # 2. 썸네일 스캔용 정적 PNG (Matplotlib)
    plt.figure(figsize=(8, 6))

    fact_data = df[df['Group'] == 'Fact']
    counter_data = df[df['Group'] == 'Counter']

    plt.scatter(fact_data['PC1'], fact_data['PC2'], alpha=0.6, label='Fact', c='blue', s=15)
    plt.scatter(counter_data['PC1'], counter_data['PC2'], alpha=0.6, label='Counter', c='red', s=15)
    
    plt.title(f'PCA 2D - Layer {layer_idx} (Top 2 Var: {sum(explained_var[:2]):.2f}%)')
    plt.xlabel(f'PC1 ({explained_var[0]:.2f}%)')
    plt.ylabel(f'PC2 ({explained_var[1]:.2f}%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    if do_show: plt.show()
    
    plt.savefig(get_file_path_pca(out_prefix, 2, 'png', layer_idx), dpi=300, bbox_inches='tight')
    plt.close() # 메모리 누수 방지

In [ ]:
def visualize_pca_3d(layer_idx, df, explained_var, out_prefix, do_show=True):
    # 1. 상세 확인용 인터랙티브 HTML (Plotly)
    fig_html = px.scatter_3d(
        df, x='PC1', y='PC2', z='PC3', color='Group',
        color_discrete_map={'Fact': 'blue', 'Counter': 'red'},
        opacity=0.7,
        title=f'PCA 3D - Layer {layer_idx} (Top 3 Var: {sum(explained_var[:3]):.2f}%)',
        labels={
            'PC1': f'PC1 ({explained_var[0]:.2f}%)',
            'PC2': f'PC2 ({explained_var[1]:.2f}%)',
            'PC3': f'PC3 ({explained_var[2]:.2f}%)'
        },
        width=800,
        height=600
    )
    fig_html.update_traces(marker=dict(size=3))
    fig_html.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig_html.write_html(get_file_path_pca(out_prefix, 3, 'html', layer_idx))

    # 2. 썸네일 스캔용 정적 PNG (Matplotlib)
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d') # 3D 축 생성

    fact_data = df[df['Group'] == 'Fact']
    counter_data = df[df['Group'] == 'Counter']

    ax.scatter(fact_data['PC1'], fact_data['PC2'], fact_data['PC3'], alpha=0.6, label='Fact', c='blue', s=15)
    ax.scatter(counter_data['PC1'], counter_data['PC2'], counter_data['PC3'], alpha=0.6, label='Counter', c='red', s=15)
    
    ax.set_title(f'PCA 3D - Layer {layer_idx} (Top 3 Var: {sum(explained_var[:3]):.2f}%)')
    ax.set_xlabel(f'PC1 ({explained_var[0]:.2f}%)')
    ax.set_ylabel(f'PC2 ({explained_var[1]:.2f}%)')
    ax.set_zlabel(f'PC3 ({explained_var[2]:.2f}%)')
    ax.legend()
    if do_show: plt.show()
    
    plt.savefig(get_file_path_pca(out_prefix, 3, 'png', layer_idx), dpi=300, bbox_inches='tight')
    plt.close() # 메모리 누수 방지

In [ ]:
def visualize_pca_mean_shift(layer_idx, delta_hs, V, out_prefix, pc_idx=0, do_show=True):
    v_pc1 = V[pc_idx]
    shifts = np.dot(delta_hs, v_pc1)
    mean_shift = np.mean(shifts)

    if mean_shift < 0:
        v_pc1 = -v_pc1
        shifts = -shifts
        mean_shift = -mean_shift

    plt.figure(figsize=(8, 6))
    seaborn.histplot(shifts, bins=50, kde=True, color='purple', alpha=0.6)
    
    plt.axvline(x=0, color='black', linestyle='--', linewidth=2, label=f'Zero Shift (No Effect)')
    plt.axvline(x=mean_shift, color='red', linestyle='-', linewidth=2, label=f'Mean Shift: {mean_shift:.2f}')

    plt.title(f'Layer {layer_idx} Distribution of Pairwise Shift along PC{pc_idx+1}', fontsize=14, fontweight='bold')
    plt.xlabel(f'Shift Magnitude towards Vulnerability Direction (PC{pc_idx+1})', fontsize=12)
    plt.ylabel('Frequency (# of Prompts)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(axis='y', linestyle=':', alpha=0.7)
    plt.tight_layout()
    if do_show: plt.show()

    out_file_path = f'{out_prefix}/pc{pc_idx+1}_mean_shift/pc{pc_idx+1}_layer_{layer_idx}_mean_shift.png'
    file_utils.make_parent(out_file_path)

    plt.savefig(out_file_path, dpi=300, bbox_inches='tight')
    plt.close()

    return mean_shift

In [ ]:
def visualize_pca_for_each_layer(num_layers, in_prefix, out_prefix, n_components=10):
    sum_explained_vars = {}

    '''
        - Layer '0'은 단순 단어 사전
            - Layer 0(hidden_states[0])은 트랜스포머의 어텐션(Attention) 블록을 타기 전의 가장 날것의 단어 임베딩(Word Embedding)
            - 프롬프트를 Question: {query}, Context: {context}, Answer: 형태로 만들었기 때문에 모든 마지막 토큰이 같음
            - 모든 데이터의 마지막 토큰이 동일하므로, Layer 0에서는 모든 데이터가 소수점 끝자리까지 완벽하게 똑같은 벡터
    '''
    for layer_idx in range(1, num_layers+1):
        # 1. 데이터 로드
        fact_file_path = f'{in_prefix}/fact/layer_{layer_idx}_h_states.pt'
        counter_file_path = f'{in_prefix}/counter/layer_{layer_idx}_h_states.pt'

        if (not file_utils.exists(fact_file_path)) or (not file_utils.exists(counter_file_path)):
            continue

        layer_h_states_fact = torch.load(fact_file_path, weights_only=True).numpy()
        layer_h_states_counter = torch.load(counter_file_path, weights_only=True).numpy()
        print(f'layer_h_states_fact shape : {layer_h_states_fact.shape}')
        print(f'layer_h_states_counter shape : {layer_h_states_counter.shape}')

        # 2. 데이터 결합 및 라벨링
        X = np.vstack([layer_h_states_fact, layer_h_states_counter])
        y = np.array(['Fact']*len(layer_h_states_fact) + ['Counter']*len(layer_h_states_counter))
        print(f'X shape : {X.shape}')
        print(f'y shape : {y.shape}')

        # 3. 데이터 스케일링 (PCA 전 필수 과정)
        X_scaled = StandardScaler().fit_transform(X)

        # 4. PCA 수행
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        print(f'X_pca shape : {X_pca.shape}\n')

        # 설명 분산 비율 출력 (데이터가 얼마나 잘 표현되었는지 확인)
        explained_var = pca.explained_variance_ratio_ * 100
        for i in range(n_components):
            print(f'\tPC{i+1} : {explained_var[i]:.4f}%')
        print()

        sum_explained_var = sum(explained_var[:3])
        sum_explained_vars[f'layer_{layer_idx}'] = sum_explained_var
        print(f'layer_{layer_idx} - PCA Explained Variance (Top 3 Sum) : {sum_explained_var:.4f}%')

        # 시각화를 위한 DF 생성
        df = pd.DataFrame({'Group': y})
        for i in range(3):
            df[f'PC{i+1}'] = X_pca[:, i]
        
        # 5. 시각화
        visualize_pca_2d(layer_idx, df, explained_var, out_prefix)
        visualize_pca_3d(layer_idx, df, explained_var, out_prefix)
    
    sorted_sum_explained_vars = container_utils.sorted_dict_value(sum_explained_vars)
    for key in sorted_sum_explained_vars.keys():
        print(f'{key} - PCA Explained Variance (sum) : {sorted_sum_explained_vars[key]:.4f}')

In [ ]:
def visualize_pca_for_each_layer_with_delta(num_layers, in_prefix, out_prefix, n_components=10):
    sum_explained_vars, mean_shifts = {}, {}
    mean_shift_pc_idx = 0

    '''
        - Layer '0'은 단순 단어 사전
            - Layer 0(hidden_states[0])은 트랜스포머의 어텐션(Attention) 블록을 타기 전의 가장 날것의 단어 임베딩(Word Embedding)
            - 프롬프트를 Question: {query}, Context: {context}, Answer: 형태로 만들었기 때문에 모든 마지막 토큰이 같음
            - 모든 데이터의 마지막 토큰이 동일하므로, Layer 0에서는 모든 데이터가 소수점 끝자리까지 완벽하게 똑같은 벡터
    '''
    for layer_idx in range(1, num_layers+1):
        # 1. 데이터 로드
        fact_file_path = f'{in_prefix}/fact/layer_{layer_idx}_h_states.pt'
        counter_file_path = f'{in_prefix}/counter/layer_{layer_idx}_h_states.pt'

        if (not file_utils.exists(fact_file_path)) or (not file_utils.exists(counter_file_path)):
            continue

        layer_h_states_fact = torch.load(fact_file_path, weights_only=True).numpy()
        layer_h_states_counter = torch.load(counter_file_path, weights_only=True).numpy()
        print(f'layer_h_states_fact shape : {layer_h_states_fact.shape}')
        print(f'layer_h_states_counter shape : {layer_h_states_counter.shape}\n')

        # 2. 차이 벡터 계산
        delta_hs = layer_h_states_counter - layer_h_states_fact
        print(f'delta_hs shape : {delta_hs.shape}')

        # 3. 데이터 스케일링 (PCA 전 필수 과정)
        scaler = StandardScaler()
        delta_scaled = scaler.fit_transform(delta_hs)

        # 4. PCA 수행
        pca = PCA(n_components=n_components)
        delta_pca = pca.fit_transform(delta_scaled)
        print(f'delta_pca shape : {delta_pca.shape}')

        # 설명 분산 비율 출력 (데이터가 얼마나 잘 표현되었는지 확인)
        explained_var = pca.explained_variance_ratio_ * 100
        for i in range(n_components):
            print(f'\tPC{i+1} : {explained_var[i]:.4f}%')
        print()

        sum_explained_var = sum(explained_var[:3])
        sum_explained_vars[f'layer_{layer_idx}'] = sum_explained_var
        print(f'layer_{layer_idx} - PCA Explained Variance (Top 3 Sum) : {sum_explained_var:.4f}%\n')

        # 5. 히든 차원과 동일한 크기의 기저 벡터(화살표)
        V = pca.components_
        print(f'V shape : {V.shape}\n')

        # 6. 기존 히든을 기저 벡터에 투영
        fact_scaled = scaler.transform(layer_h_states_fact)
        counter_scaled = scaler.transform(layer_h_states_counter)
        print(f'fact_scaled shape : {fact_scaled.shape}')
        print(f'counter_scaled shape : {counter_scaled.shape}\n')

        coord_fact = np.dot(fact_scaled, V.T)
        coord_counter = np.dot(counter_scaled, V.T)
        print(f'coord_fact shape : {coord_fact.shape}')
        print(f'coord_counter shape : {coord_counter.shape}\n')

        # 시각화를 위한 DF 생성
        coords_combined = np.vstack([coord_fact, coord_counter])
        y = np.array(['Fact']*len(coord_fact) + ['Counter']*len(coord_counter))

        df = pd.DataFrame({'Group': y})
        for i in range(3):
            df[f'PC{i+1}'] = coords_combined[:, i]
        
        # 7. 시각화
        visualize_pca_2d(layer_idx, df, explained_var, out_prefix)
        visualize_pca_3d(layer_idx, df, explained_var, out_prefix)

        mean_shift = visualize_pca_mean_shift(layer_idx, delta_hs, V, out_prefix, mean_shift_pc_idx)
        mean_shifts[f'layer_{layer_idx}'] = mean_shift
        print(f'layer_{layer_idx} - PC{mean_shift_pc_idx+1} Mean Shift : {mean_shift}\n')



    for key in container_utils.sorted_dict_value(sum_explained_vars).keys():
        print(f'{key} - PCA Explained Variance (sum) : {sum_explained_vars[key]:.4f}')

    for key in container_utils.sorted_dict_value(mean_shifts).keys():
        print(f'{key} - PC{mean_shift_pc_idx+1} Mean Shift : {mean_shifts[key]:.4f}')
    mean_shift_avg = np.mean(list(mean_shifts.values()))
    print(f'PC{mean_shift_pc_idx+1} Mean Shift Avg : {mean_shift_avg:.4f}\n')

    return mean_shift_avg

In [ ]:
# extract_size_pairs = []

# for extract_fact_size in range(1, 11):
#     for extract_counter_size in range(1, 11):
#         extract_size_pairs.append([extract_fact_size, extract_counter_size])

extract_size_pairs = [[2,8], [8,2], [3,7], [7,3], [1,3], [3,1]]

context_orders = ['shuffle']

In [ ]:
mean_shift_avgs = {}

for extract_size_pair in extract_size_pairs:
    extract_fact_size, extract_counter_size = extract_size_pair[0], extract_size_pair[1]

    for context_order in context_orders:
        print(f'extract_fact_size : {extract_fact_size}, extract_counter_size : {extract_counter_size}, context_order : {context_order}\n')

        model_name, num_layers = 'Llama-3.2-3B', 28
        in_prefix = f'{in_dir}/{model_name}/{context_order}/F-{extract_fact_size}_C-{extract_counter_size}'
        out_prefix = f'{out_dir}/{model_name}/{context_order}/F-{extract_fact_size}_C-{extract_counter_size}'
        
        mean_shift_avg = visualize_pca_for_each_layer_with_delta(num_layers, in_prefix, out_prefix)

        key = f'{model_name}_{context_order}_F-{extract_fact_size}_C-{extract_counter_size}'
        mean_shift_avgs[key] = mean_shift_avg

for key in mean_shift_avgs.keys():
    print(f'{key} Mean Shift Avg : {mean_shift_avgs[key]}')